        # 🧩 L08　函式與模組
        **Python 冒險之旅 2026**　｜　Day 4（09/03 四）🏔️ 函式之島　｜　關卡　｜　🏅 100 XP

        📖 對應教科書：第 7 章 7.1–7.7


        ### 🎯 這一關你會學到
        - 用 def 定義函式、return 傳回值
- 參數預設值與關鍵字引數
- 使用 math / random / datetime 等內建模組

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/python-quest-2026/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  Python 冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins

_LEVEL = "L08"
_SALT = "python-quest-2026-datama"
_TASKS = ["8-1", "8-2", "8-3", "8-4", "8-5", "8-6", "8-7"]
_XP_EACH = 14
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_pyquest_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

def 行列表(out):
    return [ln.rstrip() for ln in str(out).splitlines() if ln.strip()]

class _NeedMoreInput(Exception):
    pass

_HIST = builtins.__dict__.setdefault("_pyquest_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_pyquest_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_pyquest_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

def _find_cell(tid):
    marker = "# 🎯 任務 " + tid
    for cell in reversed(_history()):
        if marker in cell:
            lines = [ln for ln in cell.splitlines()
                     if not re.match(r"\s*(檢查|通關密語)\s*\(", ln)]
            return "\n".join(lines)
    return None

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                _plt.show = _orig_show
        return buf.getvalue(), ns
    run.src = src
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _progress():
    done = sum(1 for t in _TASKS if _PASSED.get(t))
    bar = "■" * done + "□" * (len(_TASKS) - done)
    return f"[{bar}] {done}/{len(_TASKS)}"

def 檢查(tid):
    tid = str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    src = _find_cell(tid)
    if src is None:
        print(f"❌ 找不到「# 🎯 任務 {tid}」的程式格。請先執行那一格（並保留第一行的標記），再執行這裡。")
        return
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        result = (False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。")
    except Exception as e:
        tb = traceback.format_exc().strip().splitlines()[-1]
        result = (False, f"程式執行時發生錯誤 → {tb}")
    ok, extra = (result, "") if isinstance(result, bool) else result
    if ok:
        first = not _PASSED.get(tid)
        _PASSED[tid] = True
        print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")
        if all(_PASSED.get(t) for t in _TASKS):
            print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
    else:
        print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
        if extra: print("   💬 " + str(extra))
        if _HINTS.get(tid): print("   💡 提示：" + _HINTS[tid])
        print("   👉 修改程式後，先重新執行任務那一格，再執行這一格。")

def 通關密語():
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_SALT}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：PYQ-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_8_1(run):
    out, ns = run()
    f = ns.get("bmi")
    if not callable(f): return (False, "要定義函式 bmi。")
    if abs(f(170, 66) - 22.837) > 0.01: return (False, "bmi(170, 66) 應該約 22.8。")
    return (abs(f(160, 50) - 19.53) < 0.01, "bmi(160, 50) 應該約 19.5。")
任務定義("8-1", _check_8_1, 提示="return w / (h / 100) ** 2。")

def _check_8_2(run):
    out, ns = run()
    f = ns.get("stats")
    if not callable(f): return (False, "要定義函式 stats。")
    if tuple(f([67, 80, 45, 50, 73])) != (315, 63.0, 80): return (False, "stats([67, 80, 45, 50, 73]) 應該傳回 (315, 63.0, 80)。")
    return (tuple(f([1, 2, 3])) == (6, 2.0, 3), "stats([1, 2, 3]) 應該傳回 (6, 2.0, 3)。")
任務定義("8-2", _check_8_2, 提示="return total, total / len(lst), max(lst)。")

def _check_8_3(run):
    out, ns = run()
    f = ns.get("greet")
    if not callable(f): return (False, "要定義函式 greet。")
    if f('小明') != '你好，小明！': return (False, "greet('小明') 應該傳回 你好，小明！（注意全形逗號與驚嘆號）。")
    return (f('Amy', 'Hi') == 'Hi，Amy！' and f(greeting='嗨', name='B') == '嗨，B！', "greet('Amy', 'Hi') 應該是 Hi，Amy！")
任務定義("8-3", _check_8_3, 提示="greeting='你好' 是預設值；return f'{greeting}，{name}！'。")

def _check_8_4(run):
    out, ns = run()
    if "global" not in run.src: return (False, "要在函式裡宣告 global count。")
    return (ns.get("count") == 3, f"呼叫 3 次後 count 應該是 3，現在是 {ns.get('count')}。")
任務定義("8-4", _check_8_4, 提示="函式第一行寫 global count。")

def _check_8_5(run):
    out, ns = run()
    import datetime as _dt
    if abs(ns.get("area", 0) - 78.5398) > 0.01: return (False, "area 應該是 math.pi * 5 ** 2 ≈ 78.54。")
    d = ns.get("dice")
    if not (isinstance(d, list) and len(d) == 5 and all(isinstance(x, int) and 1 <= x <= 10 for x in d)): return (False, "dice 要是 5 個 1～10 的整數。")
    return (ns.get("year") == _dt.datetime.now().year, "year 應該是今年（datetime.datetime.now().year）。")
任務定義("8-5", _check_8_5, 提示="math.pi * 5 ** 2；random.randint(1, 10)；datetime.datetime.now().year。")

def _check_8_6(run):
    out, ns = run("48", "72")
    f = ns.get("gcd")
    if not callable(f): return (False, "要定義函式 gcd。")
    if f(48, 72) != 24 or f(17, 5) != 1 or f(100, 75) != 25: return (False, "gcd(48,72)=24、gcd(17,5)=1、gcd(100,75)=25。")
    return (出現(out, "GCD為24"), "輸入 48、72 應該印出 GCD為 24。")
任務定義("8-6", _check_8_6, 提示="if b == 0: return a；否則 return gcd(b, a % b)。")

def _check_8_7(run):
    out, ns = run("1000000", "2.2", "3")
    f = ns.get("compound")
    if not callable(f): return (False, "要定義函式 compound。")
    if abs(f(1000000, 2.2, 3) - 1068162.2) > 1: return (False, "compound(1000000, 2.2, 3) 應該約 1068162.2。")
    return (出現(out, "1068162.2"), "輸出要顯示 1068162.2。")
任務定義("8-7", _check_8_7, 提示="p * (1 + rate / 100 / 12) ** (12 * years)。")


## 🧩 8-1　函式：把常用的功能打包起來
```python
def 函式名稱(參數1, 參數2):     # 定義（def = define）
    敘述
    return 傳回值               # 可有可無
結果 = 函式名稱(引數1, 引數2)    # 呼叫
```
- 先定義、後呼叫。定義時不會執行，呼叫時才執行。
- `return` 會把值**傳回**呼叫處，並結束函式；可以一次傳回多個值（其實是元組）。
- 沒有 `return` 的函式傳回 `None`。

In [ ]:
def average(n1, n2):                 # 課本 ex07/function01.py
    a = (n1 + n2) / 2
    return a

print(average(80, 90))
avg = average(3, 4)
print(f'平均為 {avg:.1f}')

def progress(a1, d, n):              # 課本 ex07/function03.py：傳回兩個值
    an = a1 + (n - 1) * d
    sn = n * (a1 + an) / 2
    return an, sn

last, total = progress(1, 2, 10)
print(last, total)

## 8-2　參數的花樣：預設值、關鍵字引數、傳串列

In [ ]:
def triangle(B=6, H=6):              # 課本 ex07/triangle02.py：預設值
    return B * H / 2
print(triangle(10, 5), triangle(10), triangle())          # 25.0 30.0 18.0
print(triangle(H=4, B=10))                                # 關鍵字引數：順序可以不同

def Triple(lst):                      # 課本 ex07/callByRef.py：串列會被改到！
    for i in range(len(lst)):
        lst[i] = lst[i] * 3
arr = [2, 4, 6]
Triple(arr)
print(arr)                            # [6, 12, 18]

## 8-3　全域變數與區域變數（課本 7.6）
函式**裡面**建立的變數是區域變數，函式結束就消失；函式外的是全域變數。
函式裡可以**讀**全域變數，但要**修改**它必須先宣告 `global`。

In [ ]:
n = 100
def subpro():
    global n
    n = n + 10           # 沒有 global 這行會出錯
    m = 20               # 區域變數
    print('subpro 裡：', n, m)
subpro()
print('外面：', n)       # 110

## 8-4　內建模組：`math`、`random`、`time`、`datetime`（課本 7.2）
用 `import 模組` 或 `import 模組 as 別名` 或 `from 模組 import 函式`。

In [ ]:
import math
print(math.pi, math.sqrt(49), math.floor(3.7), math.ceil(3.2), math.pow(2, 10))
import random as R
print(R.randint(1, 6))                 # 1~6 的整數（含兩端）
print(R.choice(['剪刀', '石頭', '布']))  # 隨機挑一個
print(R.sample(range(1, 50), 6))       # 不重複抽 6 個
import datetime as DT
now = DT.datetime.now()
print(now.year, now.month, now.day, f'{now:%Y/%m/%d %A}')
import time
t1 = time.time(); time.sleep(0.5); print(f'暫停了 {time.time() - t1:.2f} 秒')

## 8-5　遞迴：函式呼叫自己（課本 7.7）
遞迴一定要有**終止條件**，否則會無限呼叫。

In [ ]:
def fact(n):                         # 課本 ex07/factorial.py
    if n <= 1:
        return 1
    return n * fact(n - 1)
print(fact(5))                       # 120

### 🎯 任務 8-1　BMI 函式

定義函式 `bmi(h, w)`（身高公分、體重公斤）傳回 BMI 值，呼叫 `bmi(170, 66)` 並印出小數 1 位。

In [ ]:
# 🎯 任務 8-1　BMI 函式（請保留這一行）
def bmi(h, w):
    ???
print(f"{bmi(170, 66):.1f}")

In [ ]:
檢查("8-1")   # ◀ 執行這一格，看看任務 8-1 有沒有過關

### 🎯 任務 8-2　多個傳回值

定義 `stats(lst)` 傳回串列的 **(總和, 平均, 最大值)** 三個值，呼叫後印出 `總和 315 平均 63.0 最大 80`。

In [ ]:
# 🎯 任務 8-2　多個傳回值（請保留這一行）
def stats(lst):
    ???
    return ???
total, avg, big = stats([67, 80, 45, 50, 73])
print("總和", total, "平均", avg, "最大", big)

In [ ]:
檢查("8-2")   # ◀ 執行這一格，看看任務 8-2 有沒有過關

### 🎯 任務 8-3　預設參數：打招呼

定義 `greet(name, greeting='你好')` 傳回 `'你好，小明！'` 這樣的字串。`greet('小明')` → `你好，小明！`；`greet('Amy', 'Hi')` → `Hi，Amy！`。印出這兩個結果。

In [ ]:
# 🎯 任務 8-3　預設參數：打招呼（請保留這一行）
def greet(name, greeting=???):
    return ???
print(greet('小明'))
print(greet('Amy', 'Hi'))

In [ ]:
檢查("8-3")   # ◀ 執行這一格，看看任務 8-3 有沒有過關

### 🎯 任務 8-4　計數器（global）

全域變數 `count = 0`。定義 `add_one()` 每呼叫一次就讓 `count` 加 1（要用 `global`）。呼叫 3 次後印出 `count`。

In [ ]:
# 🎯 任務 8-4　計數器（global）（請保留這一行）
count = 0
def add_one():
    ???
    count += 1
add_one(); add_one(); add_one()
print(count)

In [ ]:
檢查("8-4")   # ◀ 執行這一格，看看任務 8-4 有沒有過關

### 🎯 任務 8-5　模組應用

完成三件事：(1) 用 `math` 算半徑 5 的圓面積 `area`（用 `math.pi`）；(2) 用 `random.randint` 產生 5 個 1～10 的整數放進串列 `dice`；(3) 用 `datetime` 取得今年年份 `year`。印出三者。

In [ ]:
# 🎯 任務 8-5　模組應用（請保留這一行）
import math, random, datetime
area = ???
dice = []
for i in range(5):
    dice.append(???)
year = ???
print(f"面積 {area:.2f}", dice, year)

In [ ]:
檢查("8-5")   # ◀ 執行這一格，看看任務 8-5 有沒有過關

### 🎯 任務 8-6　最大公因數（遞迴或迴圈）

定義 `gcd(a, b)` 傳回最大公因數（歐幾里得：`gcd(a, b) = gcd(b, a % b)`，`b == 0` 時答案是 `a`）。讀取兩個正整數並印出 `a, b 兩整數的GCD為 24`。

In [ ]:
# 🎯 任務 8-6　最大公因數（遞迴或迴圈）（請保留這一行）
def gcd(a, b):
    ???
a = int(input('輸入第一個正整數 a：'))
b = int(input('輸入第二個正整數 b：'))
print(f"a, b 兩整數的GCD為 {gcd(a, b)}")

In [ ]:
檢查("8-6")   # ◀ 執行這一格，看看任務 8-6 有沒有過關

### 🎯 任務 8-7　複利試算函式

定義 `compound(p, rate, years)`：每月計息，傳回 n 年後本利和 `p * (1 + rate/100/12) ** (12*years)`。讀取本金、年利率(%)、年數，印出 `*** 3 年後領回本利和：1068162.2 ***`（小數 1 位）。

In [ ]:
# 🎯 任務 8-7　複利試算函式（請保留這一行）
def compound(p, rate, years):
    return ???
p = int(input('請輸入本金：'))
rate = float(input('請輸入年利率(％)：'))
years = int(input('幾年後領回：'))
print(f"*** {years} 年後領回本利和：{compound(p, rate, years):.1f} ***")

In [ ]:
檢查("8-7")   # ◀ 執行這一格，看看任務 8-7 有沒有過關

---
## 🔑 通關密語

全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：🗂️ L09 元組、字典、集合** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/python-quest-2026/blob/main/notebooks/L09_tuple_dict_set.ipynb)

回到入口網頁：https://johnnychao.github.io/python-quest-2026/